# v9 - Division-Aware Tracking - Phase 0 (oracle audit / mandatory gate)

Plan: `docs/v9_division_aware_tracking_plan.md`.

**What this notebook does (Phase 0).** It (optionally) GENERATES baseline predicted graphs
(`.geff`) with the frozen detector+transformer+ILP on a held-out subset, then measures - with the
**official scorer** - the ceiling of division recovery under three candidate families:

- **A** orphan-only: add `M -> D2` where `D2` has no incoming edge.
- **B** steal a weak child (scaffold).
- **C** joint daughter selection (scaffold).

For each family it reports, per specimen (`44b6` / `6bba`): recoverable GT divisions, oracle
`div_J`, `Delta adj`, `Delta score = Delta adj + 0.1 * Delta div_J`. **GATE 0** then decides whether
to build Phase 1.

**How to run.** GPU ON. Attach: (1) competition data (train `<id>.zarr` + `<id>.geff`), (2) the
support-pack dataset (`repo/` + `weights/` + `wheels/`). Internet ON is fine (local audit, not a
submission). Cell 0.2b does the one GPU pass; everything after it is CPU and fast.

**Scope note.** The gate needs only `pred .geff + GT + scorer`. The pre-ILP candidate-edge export
(transformer logits/ranks/margins) is a Phase-1 feature concern (final optional cell). Absolute
local numbers are optimistic (audit runs on the split_0 held-out test, but the extractors still saw
all train videos); trust per-specimen consistency and deltas, not absolutes.

In [1]:
# --- Phase 0.1 : environment ---------------------------------------------------
import importlib, os, sys, glob, subprocess
from pathlib import Path

# Polars 1.x is split into a Python package plus a compiled runtime wheel.
# Prefer the broadly compatible runtime on Kaggle CPUs.
os.environ.setdefault("POLARS_PREFER_PKG", "32")

def _find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

# Support-pack repo = the dir that contains scripts/predict_unet_transformer.py
_hits = _find("/kaggle/input/**/scripts/predict_unet_transformer.py")
assert _hits, "support-pack repo not found under /kaggle/input (attach the pack dataset)."
REPO = Path(_hits[0]).parent.parent
for p in (REPO / "src", REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("REPO =", REPO)

# Dependency resolution is deliberately disabled so pip cannot replace Kaggle's
# already-imported numpy/scipy and create a binary ABI mismatch. Because --no-deps
# is used, every runtime dependency must be named explicitly. This list is the
# dependency closure validated by the v7 offline gate and the v8 submission.
PIP_SPECS = [
    "tracksdata", "pyscipopt", "ilpy>=0.5.1",
    "zarr>=3.0.10,<4", "geff>=1.1.3.1.1", "geff-spec<1.2",
    "polars>=1.36", "polars-runtime-32>=1.36",
    "blosc2", "dask", "imagecodecs", "scikit-image>=0.24",
    "pyarrow", "rustworkx>=0.17.1", "sqlalchemy>=2", "numcodecs>=0.13,<0.16",
    "donfig>=0.8", "google-crc32c>=1.5", "bidict>=0.23.1", "psygnal>=0.14",
    "rich", "networkx>=3.2.1", "pydantic>=2.11", "pydantic-core",
    "annotated-types", "typing-extensions>=4.13", "typing-inspection",
    "markdown-it-py", "pygments", "click", "cloudpickle", "fsspec",
    "partd", "locket", "toolz", "pyyaml", "ndindex", "msgpack",
    "numexpr", "deprecated", "wrapt",
]

CRITICAL_MODULES = ("tracksdata", "geff", "geff_spec", "zarr",
                    "pyscipopt", "ilpy", "donfig", "numcodecs",
                    "polars", "blosc2", "dask", "imagecodecs",
                    "pyarrow", "rustworkx", "sqlalchemy", "skimage")

def _clear_partial_imports():
    # A failed import can leave half-initialized packages in sys.modules.
    roots = set(CRITICAL_MODULES) | {"skimage"}
    for name in list(sys.modules):
        if any(name == root or name.startswith(root + ".") for root in roots):
            sys.modules.pop(name, None)
    importlib.invalidate_caches()

def _polars_binary_ok():
    # The real test: a py3-none-any polars wheel imports fine but has NO compiled
    # binary ('Polars binary is missing!') -> constructing ANY DataFrame raises
    # `PyDataFrame is not defined`. A Series/DataFrame build catches that; a bare
    # import does not.
    try:
        import polars as _pl
        _pl.DataFrame({"_x": [1]})
        return True
    except Exception:
        return False

def _import_failures():
    failures = {}
    for name in CRITICAL_MODULES:
        try:
            importlib.import_module(name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    try:
        import zarr as _zarr
        if int(_zarr.__version__.split(".")[0]) < 3:
            failures["zarr"] = f"zarr {_zarr.__version__} is too old; need >=3"
    except Exception:
        pass
    try:
        import polars as _pl
        polars_ok = (hasattr(_pl, "Float16") and _polars_binary_ok())
        if not polars_ok:
            failures["polars"] = f"polars {_pl.__version__} unusable (old or binary missing)"
    except Exception:
        pass
    return failures

def _run_pip(command, label):
    print(label)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return result.returncode == 0

failures = _import_failures()
if failures:
    print("Dependency check failed:", failures)
    wheel_dirs = []
    for path in [REPO.parent / "wheels", *map(Path, _find("/kaggle/input/**/wheels"))]:
        if path.is_dir() and path not in wheel_dirs:
            wheel_dirs.append(path)

    base = [sys.executable, "-m", "pip", "install", "--no-deps"]
    installed = False
    if wheel_dirs:
        offline = base + ["--no-index"]
        for path in wheel_dirs:
            offline += ["--find-links", str(path)]
        installed = _run_pip(offline + PIP_SPECS,
                             f"Installing from offline wheels: {wheel_dirs}")
    if not installed:
        installed = _run_pip(base + PIP_SPECS, "Offline install unavailable; trying PyPI")
    if not installed:
        raise RuntimeError("Dependency installation failed; see pip output above.")

    _clear_partial_imports()
    failures = _import_failures()
    non_polars_failures = {k: v for k, v in failures.items() if k != "polars"}
    if non_polars_failures:
        raise ImportError(
            f"Dependencies still fail after installation: {non_polars_failures}"
        )

# polars binary guard: the offline wheels can carry a binary-less polars
# (polars-*-py3-none-any.whl) that SHADOWS Kaggle's working build once installed
# -> "Polars binary is missing!" -> `PyDataFrame is not defined` on every polars
# AND tracksdata DataFrame op. Install the Python package and its compiled runtime
# as a matched pair. Prefer the attached offline wheels, then use PyPI (this audit
# runs with internet ON). No-op when polars already works.
if not _polars_binary_ok():
    print("polars binary missing -> reinstalling polars + polars-runtime-32")
    _clear_partial_imports()
    polars_specs = ["polars>=1.36", "polars-runtime-32>=1.36"]
    repaired = False
    if wheel_dirs:
        command = [sys.executable, "-m", "pip", "install", "--no-deps",
                   "--force-reinstall", "--no-index"]
        for path in wheel_dirs:
            command += ["--find-links", str(path)]
        repaired = _run_pip(command + polars_specs,
                            "Repairing polars from offline wheels")
        _clear_partial_imports()
        repaired = repaired and _polars_binary_ok()
    if not repaired:
        repaired = _run_pip(
            [sys.executable, "-m", "pip", "install", "--no-deps",
             "--force-reinstall", *polars_specs],
            "Repairing polars from PyPI",
        )
    _clear_partial_imports()
    if not repaired or not _polars_binary_ok():
        raise ImportError(
            "polars still has no compiled runtime after reinstall; "
            "check that polars and polars-runtime-32 wheels have matching versions"
        )

import tracksdata, geff, zarr  # noqa: F401
import polars as _pl_check
print("tracksdata", getattr(tracksdata, "__version__", "?"),
      "| geff", getattr(geff, "__version__", "?"),
      "| zarr", getattr(zarr, "__version__", "?"),
      "| polars", _pl_check.__version__, "(binary OK)")

REPO = /kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/repo
Dependency check failed: {'tracksdata': "ModuleNotFoundError: No module named 'tracksdata'", 'geff': "ModuleNotFoundError: No module named 'geff'", 'geff_spec': "ModuleNotFoundError: No module named 'geff_spec'", 'zarr': "ModuleNotFoundError: No module named 'zarr'", 'pyscipopt': "ModuleNotFoundError: No module named 'pyscipopt'", 'ilpy': "ModuleNotFoundError: No module named 'ilpy'", 'donfig': "ModuleNotFoundError: No module named 'donfig'", 'numcodecs': "ModuleNotFoundError: No module named 'numcodecs'", 'imagecodecs': "ModuleNotFoundError: No module named 'imagecodecs'", 'rustworkx': "ModuleNotFoundError: No module named 'rustworkx'", 'polars': 'polars 1.35.2 unusable (old or binary missing)'}
Installing from offline wheels: [PosixPath('/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/wheels'), PosixPath('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels')]


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


In [2]:
# --- Phase 0.2 : imports -------------------------------------------------------
import json
import numpy as np
import polars as pl
import tracksdata as td
from collections import defaultdict, Counter

try:
    from geff import GeffMetadata
except Exception:
    GeffMetadata = None

from biohub_tracking.io import open_dataset, save_graph
from biohub_tracking.metrics import evaluate as compute_metric, per_sample_metrics
from predict_unet_transformer import load_model, predict_video, build_graph, PredictConfig

K = td.DEFAULT_ATTR_KEYS
print("attr keys:", K.NODE_ID, K.T, K.EDGE_SOURCE, K.EDGE_TARGET, K.MATCHED_NODE_ID)

def specimen_of(name):
    return name.split("_")[0]

attr keys: node_id t source_id target_id match_node_id


In [3]:
# --- Phase 0.2b : GENERATE baseline .geff (GPU, one pass) --------------------
# Runs the frozen detector+transformer+ILP on a held-out subset (split_0 test,
# division-richest) and saves per-video .geff to /kaggle/working/preds.
# Set GENERATE_BASELINE=False if you already have baseline predictions to audit
# (then set PRED_DIR in cell 0.3).
import torch

GENERATE_BASELINE  = True
GEN_DIR            = Path("/kaggle/working/preds"); GEN_DIR.mkdir(parents=True, exist_ok=True)
GEN_N_PER_SPECIMEN = 8          # videos per specimen to generate + audit
DET_THRESHOLD      = 0.96875    # matches the shipped 0.912 detection threshold
PICK_DIVISION_RICH = True       # rank candidates by GT division count (richer oracle signal)

# train data dir = the /kaggle/input dir with the MOST matched <id>.zarr + <id>.geff
# pairs. NOTE: glob("/kaggle/input/**/*.zarr") matches BOTH the competition train/
# AND test/ dirs; test/ has zarr but no GT geff and sorts BEFORE train alphabetically,
# so a naive sorted()[0] wrongly selects test (0 usable pairs). Rank by pair count.
def _pair_count(d):
    d = Path(d)
    return sum(1 for z in d.glob("*.zarr") if (d / f"{z.stem}.geff").exists())
_zdirs = sorted({str(Path(p).parent) for p in _find("/kaggle/input/**/*.zarr")},
                key=_pair_count, reverse=True)
DATA_DIR = Path(_zdirs[0]) if _zdirs and _pair_count(_zdirs[0]) > 0 else None
assert DATA_DIR is not None, "train data dir (with matched <id>.zarr + <id>.geff pairs) not found."
print("DATA_DIR =", DATA_DIR, "|", _pair_count(DATA_DIR), "zarr+geff pairs")

def _gt_div_count(name):
    try:
        gt = open_dataset(DATA_DIR / name, require_tracks=True).tracks
        if gt.num_edges() == 0:
            return 0
        df = gt.edge_attrs(attr_keys=[K.EDGE_SOURCE])
        c = Counter(int(s) for s in df[K.EDGE_SOURCE].to_list())
        return sum(1 for v in c.values() if v >= 2)
    except Exception:
        return 0

if GENERATE_BASELINE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _w = [w for w in _find("/kaggle/input/**/edge_predictor_best.pth")
          if (Path(w).parent / "config.json").exists()]
    assert _w, "edge_predictor_best.pth (with sibling config.json) not found."
    WEIGHTS = Path(_w[0]); print("WEIGHTS =", WEIGHTS, "| device =", device)
    model, window_size, downsample = load_model(WEIGHTS, device)

    # candidate videos = split_0 test (held out for THIS model) if the splits file exists
    splits_file = DATA_DIR / "dataset_splits.json"
    if splits_file.exists():
        cand = json.loads(splits_file.read_text())[0]["test"]
        print("using split_0 test:", len(cand), "videos")
    else:
        cand = [p.stem for p in sorted(DATA_DIR.glob("*.zarr"))]
        print("no splits file -> using all:", len(cand), "videos")
    cand = [n for n in cand
            if (DATA_DIR / f"{n}.geff").exists() and (DATA_DIR / f"{n}.zarr").exists()]

    by = defaultdict(list)
    if PICK_DIVISION_RICH:
        for dc, n in sorted(((_gt_div_count(n), n) for n in cand), reverse=True):
            by[specimen_of(n)].append(n)
    else:
        for n in cand:
            by[specimen_of(n)].append(n)
    GEN_NAMES = [n for s in sorted(by) for n in by[s][:GEN_N_PER_SPECIMEN]]
    print("generating", len(GEN_NAMES), "videos:",
          {s: sum(specimen_of(n) == s for n in GEN_NAMES) for s in set(map(specimen_of, GEN_NAMES))})

    cfg = PredictConfig(
        det_threshold=DET_THRESHOLD, det_tta=True,
        edge_activation="softmax", threshold=0.5,
        use_ilp=True, ilp_edge_weight=-1.0,
        ilp_appearance_weight=0.1, ilp_disappearance_weight=0.1,
        ilp_division_weight=1.0,
    )
    for name in GEN_NAMES:
        out = GEN_DIR / f"{name}.geff"
        if out.exists():
            print("  skip (exists):", name); continue
        coords, edges = predict_video(model, DATA_DIR / name, device, cfg=cfg,
                                      window_size=window_size, downsample=downsample)
        g = build_graph(coords, edges)
        if cfg.use_ilp and g.num_edges() > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=cfg.ilp_edge_weight * td.EdgeAttr("edge_prob"),
                appearance_weight=cfg.ilp_appearance_weight,
                disappearance_weight=cfg.ilp_disappearance_weight,
                division_weight=cfg.ilp_division_weight,
            )
            g = solver.solve(g)
        save_graph(g, out)
        print("  saved", name, "| nodes", g.num_nodes(), "edges", g.num_edges())
    PRED_DIR = GEN_DIR
    print("PRED_DIR =", PRED_DIR)

DATA_DIR = /kaggle/input/competitions/biohub-cell-tracking-during-development/train | 199 zarr+geff pairs
WEIGHTS = /kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/weights/unet_transformer/split_0/edge_predictor_best.pth | device = cuda
no splits file -> using all: 199 videos
generating 16 videos: {'44b6': 8, '6bba': 8}


[08/26/26 02:54:28] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=921939;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=33593;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

  saved 44b6_e28840c6 | nodes 27142 edges 23007


[08/26/26 02:56:08] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=419378;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=727957;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

  saved 44b6_d5e7d891 | nodes 44382 edges 38210


...

  saved 6bba_09961292 | nodes 31741 edges 29267


[08/26/26 03:13:53] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=321188;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=655017;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

  saved 6bba_bb9f20c3 | nodes 24218 edges 22765


[08/26/26 03:15:18] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=427579;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=961963;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

  saved 6bba_969618f6 | nodes 16600 edges 15174
PRED_DIR = /kaggle/working/preds


In [4]:
# --- Phase 0.3 : config / video list ----------------------------------------
VOXEL_SCALE = (1.625, 0.40625, 0.40625)   # (z, y, x) um
MAX_DIST    = 7.0                           # node-match gate (um)
DIV_WEIGHT  = 0.1                           # metric division weight
SUBSET_PER_SPECIMEN = None                  # None = use all generated videos

# DATA_DIR / PRED_DIR normally come from cell 0.2b. If you skipped generation,
# set PRED_DIR to your baseline *.geff folder (and DATA_DIR to the train dir).
try:
    DATA_DIR
except NameError:
    # pick the train dir by matched <id>.zarr+<id>.geff pair count (see cell 0.2b:
    # test/ sorts before train/ but has no GT geff, so sorted()[0] is wrong).
    def _pc(d):
        d = Path(d)
        return sum(1 for z in d.glob("*.zarr") if (d / f"{z.stem}.geff").exists())
    _z = sorted({str(Path(p).parent) for p in _find("/kaggle/input/**/*.zarr")},
                key=_pc, reverse=True)
    DATA_DIR = Path(_z[0]) if _z and _pc(_z[0]) > 0 else None
try:
    PRED_DIR
except NameError:
    PRED_DIR = None
if PRED_DIR is None:
    _p = [d for d in sorted({str(Path(p).parent) for p in _find("/kaggle/input/**/*.geff")})
          if DATA_DIR is None or Path(d) != DATA_DIR]
    PRED_DIR = Path(_p[0]) if _p else None
assert DATA_DIR is not None and PRED_DIR is not None, "set DATA_DIR and PRED_DIR"
print("DATA_DIR =", DATA_DIR); print("PRED_DIR =", PRED_DIR)

names = [p.stem for p in sorted(Path(PRED_DIR).glob("*.geff"))
         if (DATA_DIR / f"{p.stem}.geff").exists()]
if SUBSET_PER_SPECIMEN:
    by = defaultdict(list)
    for n in names:
        by[specimen_of(n)].append(n)
    names = [n for g in by.values() for n in g[:SUBSET_PER_SPECIMEN]]
print(f"{len(names)} videos:",
      {s: sum(specimen_of(n) == s for n in names) for s in set(map(specimen_of, names))})

DATA_DIR = /kaggle/input/competitions/biohub-cell-tracking-during-development/train
PRED_DIR = /kaggle/working/preds
16 videos: {'44b6': 8, '6bba': 8}


In [5]:
# --- Phase 0.4 : graph <-> arrays, fresh-rebuild, scoring ---------------------
# Everything is kept in index space (node index 0..N-1 == row in coords) so a
# fresh scoring copy can be rebuilt cheaply (compute_metric mutates its input by
# matching, so each score needs a fresh graph). make_graph returns the node_ids
# so matches can be mapped back to indices regardless of id assignment.

_NODE_KEYSETS = [
    [K.NODE_ID, K.T, K.Z, K.Y, K.X],
    [K.NODE_ID, "t", "z", "y", "x"],
]

def _node_arrays(g):
    last = None
    for ks in _NODE_KEYSETS:
        try:
            df = g.node_attrs(attr_keys=ks)
            return (np.asarray(df[ks[0]].to_list()),
                    np.asarray(df[ks[1]].to_list(), dtype=np.float64),
                    np.asarray(df[ks[2]].to_list(), dtype=np.float64),
                    np.asarray(df[ks[3]].to_list(), dtype=np.float64),
                    np.asarray(df[ks[4]].to_list(), dtype=np.float64))
        except Exception as e:
            last = e
    raise last

def _edge_pairs(g):
    if g.num_edges() == 0:
        return []
    df = g.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
    return list(zip(df[K.EDGE_SOURCE].to_list(), df[K.EDGE_TARGET].to_list()))

def load_graph_arrays(g):
    """Graph -> (coords[N,4]=t,z,y,x, idx_edges list[(i,j)])."""
    nid, t, z, y, x = _node_arrays(g)
    id2i = {int(v): i for i, v in enumerate(nid)}
    coords = np.stack([t, z, y, x], axis=1)
    idx_edges = [(id2i[int(s)], id2i[int(tt)])
                 for s, tt in _edge_pairs(g)
                 if int(s) in id2i and int(tt) in id2i]
    return coords, idx_edges

def make_graph(coords, idx_edges):
    """Build a fresh tracksdata graph in index space; returns (graph, node_ids)."""
    g = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        g.add_node_attr_key(key, pl.Float64, -999999.0)
    node_ids = g.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coords
    ])
    if idx_edges:
        g.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        g.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        g.bulk_add_edges([
            {"source_id": node_ids[i], "target_id": node_ids[j],
             "edge_prob": 1.0,
             "edge_dist": float(np.linalg.norm((coords[i, 1:] - coords[j, 1:]) * np.asarray(VOXEL_SCALE)))}
            for (i, j) in idx_edges
        ])
    return g, node_ids

def score(coords, idx_edges, gt_graph, n_total, want_match=False):
    """Score an index-space graph vs GT. Returns metric dict; optionally the
    gt_node_id -> pred_index matching (from the scorer's own assignment)."""
    g, node_ids = make_graph(coords, idx_edges)
    id2idx = {int(nid): i for i, nid in enumerate(node_ids)}
    er = compute_metric(g, gt_graph, scale=VOXEL_SCALE, max_distance=MAX_DIST)
    d_tp, d_fp, d_fn = er.division_tp, er.division_fp, er.division_fn
    denom = d_tp + d_fp + d_fn
    div_j = (d_tp / denom) if denom > 0 else 0.0
    na = g.node_attrs(attr_keys=[K.NODE_ID, K.MATCHED_NODE_ID])
    pids = na[K.NODE_ID].to_list()
    mids = na[K.MATCHED_NODE_ID].to_list()
    matched_gt = [int(m) for m in mids if m is not None and int(m) != -1]
    recall = len(set(matched_gt)) / max(1, gt_graph.num_nodes())
    psm = per_sample_metrics(er, n_total, recall)
    adj = psm["adj_edge_jaccard"]
    out = {
        "adj": adj, "edge_j": psm["edge_jaccard"], "div_j": div_j,
        "score": (adj if adj == adj else 0.0) + DIV_WEIGHT * div_j,
        "e_tp": er.edge_tp, "e_fp": er.edge_fp, "e_fn": er.edge_fn,
        "d_tp": d_tp, "d_fp": d_fp, "d_fn": d_fn,
        "recall": recall, "n_pred": er.num_pred_nodes,
    }
    if want_match:
        gt2pred = {}
        for nid, m in zip(pids, mids):
            if m is not None and int(m) != -1:
                gt2pred[int(m)] = id2idx[int(nid)]
        out["gt2pred"] = gt2pred
    return out

def read_n_total(name):
    if GeffMetadata is None:
        return float("nan")
    try:
        meta = GeffMetadata.read(DATA_DIR / f"{name}.geff")
        v = (meta.extra or {}).get("estimated_number_of_nodes")
        return float(v) if v is not None else float("nan")
    except Exception:
        return float("nan")

print("helpers ready")

helpers ready


In [6]:
# --- Phase 0.5 : load baselines + GT, extract GT divisions --------------------
def gt_adjacency(gt):
    children, parents = defaultdict(list), defaultdict(list)
    if gt.num_edges() > 0:
        df = gt.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
        for s, t in zip(df[K.EDGE_SOURCE].to_list(), df[K.EDGE_TARGET].to_list()):
            children[int(s)].append(int(t))
            parents[int(t)].append(int(s))
    return children, parents

videos = {}
for name in names:
    pg = td.graph.IndexedRXGraph.from_geff(str(Path(PRED_DIR) / f"{name}.geff"))
    if isinstance(pg, tuple):
        pg = pg[0]
    coords, base_edges = load_graph_arrays(pg)
    gt = open_dataset(DATA_DIR / name, require_tracks=True).tracks
    ch, pa = gt_adjacency(gt)
    gt_divs = [m for m in ch if len(ch[m]) >= 2]  # GT dividing nodes (out-degree >= 2)
    videos[name] = dict(coords=coords, base_edges=base_edges, gt=gt,
                        n_total=read_n_total(name), children=ch, parents=pa, gt_divs=gt_divs)
    print(f"{name}: pred_nodes={len(coords)} pred_edges={len(base_edges)} "
          f"gt_nodes={gt.num_nodes()} gt_divs={len(gt_divs)}")
print("total GT divisions:", sum(len(v['gt_divs']) for v in videos.values()))

44b6_a21120c2: pred_nodes=21997 pred_edges=20733 gt_nodes=296 gt_divs=2
44b6_c50204e0: pred_nodes=44106 pred_edges=37170 gt_nodes=333 gt_divs=2
44b6_d2f34f90: pred_nodes=16084 pred_edges=13782 gt_nodes=283 gt_divs=2
44b6_d5e7d891: pred_nodes=44382 pred_edges=38210 gt_nodes=906 gt_divs=2
44b6_d754aa59: pred_nodes=5983 pred_edges=5552 gt_nodes=72 gt_divs=1
44b6_deabac95: pred_nodes=16126 pred_edges=14736 gt_nodes=126 gt_divs=1
44b6_e28840c6: pred_nodes=27142 pred_edges=23007 gt_nodes=311 gt_divs=2
44b6_eb2880fc: pred_nodes=42521 pred_edges=37956 gt_nodes=390 gt_divs=1
6bba_09961292: pred_nodes=31741 pred_edges=29267 gt_nodes=1950 gt_divs=4
6bba_48816121: pred_nodes=25897 pred_edges=24197 gt_nodes=935 gt_divs=5
6bba_969618f6: pred_nodes=16600 pred_edges=15174 gt_nodes=679 gt_divs=3
6bba_afb141ff: pred_nodes=5990 pred_edges=5390 gt_nodes=654 gt_divs=4
6bba_bb9f20c3: pred_nodes=24218 pred_edges=22765 gt_nodes=1925 gt_divs=3
6bba_cdcfe533: pred_nodes=28836 pred_edges=27260 gt_nodes=1419 gt_d

In [7]:
# --- Phase 0.6 : per-video baseline score + scorer matching ------------------
# One compute_metric per video gives (a) the baseline metric and (b) the exact
# gt_node -> pred_index matching the official scorer uses (so "detectable" is
# defined by the metric itself, not a home-grown NN gate).
for name, v in videos.items():
    base = score(v["coords"], v["base_edges"], v["gt"], v["n_total"], want_match=True)
    v["base"] = base
    v["gt2pred"] = base["gt2pred"]
    indeg = defaultdict(int)
    outadj = defaultdict(list)
    for (i, j) in v["base_edges"]:
        indeg[j] += 1
        outadj[i].append(j)
    v["indeg"], v["outadj"] = indeg, outadj

print("baseline (per video): adj / div_J / d_tp,fp,fn")
for name, v in videos.items():
    b = v["base"]
    print(f"  {name}: adj={b['adj']:.4f} div_J={b['div_j']:.4f} "
          f"(TP{b['d_tp']}/FP{b['d_fp']}/FN{b['d_fn']}) recall={b['recall']:.4f}")

baseline (per video): adj / div_J / d_tp,fp,fn
  44b6_a21120c2: adj=0.9562 div_J=0.0000 (TP0/FP0/FN2) recall=0.9966
  44b6_c50204e0: adj=0.7709 div_J=0.0000 (TP0/FP0/FN2) recall=0.9910
  44b6_d2f34f90: adj=0.9106 div_J=0.0000 (TP0/FP0/FN2) recall=0.9965
  44b6_d5e7d891: adj=0.7761 div_J=0.0000 (TP0/FP0/FN2) recall=0.9901
  44b6_d754aa59: adj=0.9702 div_J=0.0000 (TP0/FP0/FN1) recall=1.0000
  44b6_deabac95: adj=0.8847 div_J=0.0000 (TP0/FP0/FN1) recall=0.9921
  44b6_e28840c6: adj=0.8801 div_J=0.0000 (TP0/FP0/FN2) recall=0.9904
  44b6_eb2880fc: adj=0.8836 div_J=0.0000 (TP0/FP0/FN1) recall=0.9949
  6bba_09961292: adj=0.8967 div_J=0.0000 (TP0/FP0/FN4) recall=0.9954
  6bba_48816121: adj=0.8973 div_J=0.0000 (TP0/FP1/FN5) recall=0.9925
  6bba_969618f6: adj=0.9394 div_J=0.0000 (TP0/FP0/FN3) recall=0.9912
  6bba_afb141ff: adj=0.9196 div_J=0.0000 (TP0/FP0/FN4) recall=0.9908
  6bba_bb9f20c3: adj=0.9106 div_J=0.0000 (TP0/FP0/FN3) recall=0.9953
  6bba_cdcfe533: adj=0.9869 div_J=0.0000 (TP0/FP0/FN4) r

In [8]:
# --- Phase 0.7 : ORACLE family A (orphan-only) -------------------------------
# For each detectable GT division (mother + both daughters matched by the scorer):
#   if the mother's pred node already links to exactly one daughter, and the other
#   daughter's pred node is an orphan (in-degree 0), the oracle-correct edit is
#   add(mother_pred -> other_daughter_pred). We apply ALL such edits per video at
#   once (the true combined ceiling) and rescore.
def family_A_edits(v):
    g2p = v["gt2pred"]
    edits, detectable = [], 0
    for m in v["gt_divs"]:
        daughters = v["children"][m][:2]
        if m not in g2p or any(d not in g2p for d in daughters):
            continue
        detectable += 1
        Mi = g2p[m]
        Ds = [g2p[d] for d in daughters]
        linked = [d for d in Ds if d in v["outadj"].get(Mi, [])]
        if len(linked) != 1:
            continue                      # precondition (A): exactly one child present
        other = [d for d in Ds if d not in linked][0]
        if v["indeg"].get(other, 0) != 0:
            continue                      # not an orphan -> family B/C territory
        edits.append((Mi, other))
    return edits, detectable

rows = []
for name, v in videos.items():
    edits, detectable = family_A_edits(v)
    new_edges = v["base_edges"] + edits
    orc = score(v["coords"], new_edges, v["gt"], v["n_total"]) if edits else v["base"]
    b = v["base"]
    rows.append(dict(
        name=name, specimen=specimen_of(name),
        gt_divs=len(v["gt_divs"]), detectable=detectable, A_edits=len(edits),
        d_adj=orc["adj"] - b["adj"], d_divj=orc["div_j"] - b["div_j"],
        d_score=orc["score"] - b["score"],
        base_adj=b["adj"], orc_adj=orc["adj"],
        base_divj=b["div_j"], orc_divj=orc["div_j"],
    ))
    print(f"  {name}: A_edits={len(edits)}/{detectable}det/{len(v['gt_divs'])}gt "
          f"Dadj={rows[-1]['d_adj']:+.4f} DdivJ={rows[-1]['d_divj']:+.4f} "
          f"Dscore={rows[-1]['d_score']:+.4f}")
famA = pl.DataFrame(rows)
famA

  44b6_a21120c2: A_edits=2/2det/2gt Dadj=+0.0069 DdivJ=+1.0000 Dscore=+0.1069
  44b6_c50204e0: A_edits=1/2det/2gt Dadj=+0.0028 DdivJ=+0.5000 Dscore=+0.0528
  44b6_d2f34f90: A_edits=2/2det/2gt Dadj=+0.0070 DdivJ=+1.0000 Dscore=+0.1070
  44b6_d5e7d891: A_edits=1/2det/2gt Dadj=+0.0010 DdivJ=+0.5000 Dscore=+0.0510
  44b6_d754aa59: A_edits=1/1det/1gt Dadj=+0.0141 DdivJ=+1.0000 Dscore=+0.1141
  44b6_deabac95: A_edits=0/1det/1gt Dadj=+0.0000 DdivJ=+0.0000 Dscore=+0.0000
  44b6_e28840c6: A_edits=1/2det/2gt Dadj=+0.0032 DdivJ=+0.5000 Dscore=+0.0532
  44b6_eb2880fc: A_edits=0/1det/1gt Dadj=+0.0000 DdivJ=+0.0000 Dscore=+0.0000
  6bba_09961292: A_edits=1/3det/4gt Dadj=+0.0005 DdivJ=+0.2500 Dscore=+0.0255
  6bba_48816121: A_edits=2/3det/5gt Dadj=+0.0021 DdivJ=+0.3333 Dscore=+0.0354
  6bba_969618f6: A_edits=1/3det/3gt Dadj=+0.0015 DdivJ=+0.3333 Dscore=+0.0348
  6bba_afb141ff: A_edits=2/4det/4gt Dadj=+0.0031 DdivJ=+0.5000 Dscore=+0.0531
  6bba_bb9f20c3: A_edits=1/3det/3gt Dadj=+0.0005 DdivJ=+0.3333 D

name,specimen,gt_divs,detectable,A_edits,d_adj,d_divj,d_score,base_adj,orc_adj,base_divj,orc_divj
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""44b6_a21120c2""","""44b6""",2,2,2,0.006904,1.0,0.106904,0.956199,0.963103,0.0,1.0
"""44b6_c50204e0""","""44b6""",2,2,1,0.002803,0.5,0.052803,0.770853,0.773656,0.0,0.5
"""44b6_d2f34f90""","""44b6""",2,2,2,0.007005,1.0,0.107005,0.910608,0.917612,0.0,1.0
"""44b6_d5e7d891""","""44b6""",2,2,1,0.001013,0.5,0.051013,0.776109,0.777122,0.0,0.5
"""44b6_d754aa59""","""44b6""",1,1,1,0.014061,1.0,0.114061,0.970236,0.984297,0.0,1.0
…,…,…,…,…,…,…,…,…,…,…,…
"""6bba_afb141ff""","""6bba""",4,4,2,0.003075,0.5,0.053075,0.919558,0.922633,0.0,0.5
"""6bba_bb9f20c3""","""6bba""",3,3,1,0.000508,0.333333,0.033842,0.910626,0.911135,0.0,0.333333
"""6bba_cdcfe533""","""6bba""",4,3,3,0.002221,0.75,0.077221,0.986898,0.989119,0.0,0.75


In [9]:
# --- Phase 0.8 : per-specimen aggregate + GATE 0 (family A) ------------------
def agg(df):
    return df.group_by("specimen").agg([
        pl.col("gt_divs").sum().alias("gt_divs"),
        pl.col("detectable").sum().alias("detectable"),
        pl.col("A_edits").sum().alias("recovered"),
        pl.col("d_adj").mean().alias("mean_d_adj"),
        pl.col("d_divj").mean().alias("mean_d_divj"),
        pl.col("d_score").mean().alias("mean_d_score"),
    ]).sort("specimen")

summary = agg(famA)
print(summary)
print()
detect_frac = famA["detectable"].sum() / max(1, famA["gt_divs"].sum())
print(f"daughter-detectability ceiling: {famA['detectable'].sum()}/{famA['gt_divs'].sum()} "
      f"= {detect_frac:.2%} of GT divisions have mother+both daughters detected")
print(f"family-A recoverable (orphan precondition): {famA['A_edits'].sum()} divisions")

both_pos = bool((summary["mean_d_score"] > 0).all())
print("\n=== GATE 0 (family A) ===")
print("PASS -> build Phase 1" if both_pos else
      "family A weak -> evaluate B/C (Phase 0.9) before deciding")
print("(discount local optimism ~0.5x before reading against the LB; require both specimens > 0.)")

shape: (2, 7)
┌──────────┬─────────┬────────────┬───────────┬────────────┬─────────────┬──────────────┐
│ specimen ┆ gt_divs ┆ detectable ┆ recovered ┆ mean_d_adj ┆ mean_d_divj ┆ mean_d_score │
│ ---      ┆ ---     ┆ ---        ┆ ---       ┆ ---        ┆ ---         ┆ ---          │
│ str      ┆ i64     ┆ i64        ┆ i64       ┆ f64        ┆ f64         ┆ f64          │
╞══════════╪═════════╪════════════╪═══════════╪════════════╪═════════════╪══════════════╡
│ 44b6     ┆ 13      ┆ 13         ┆ 8         ┆ 0.004369   ┆ 0.5625      ┆ 0.060619     │
│ 6bba     ┆ 31      ┆ 26         ┆ 17        ┆ 0.002467   ┆ 0.53125     ┆ 0.055592     │
└──────────┴─────────┴────────────┴───────────┴────────────┴─────────────┴──────────────┘

daughter-detectability ceiling: 39/44 = 88.64% of GT divisions have mother+both daughters detected
family-A recoverable (orphan precondition): 25 divisions

=== GATE 0 (family A) ===
PASS -> build Phase 1
(discount local optimism ~0.5x before reading against the LB

## Phase 0.9 - families B / C (scaffold)

Family A alone is bounded by how often the true daughter is left an **orphan** by the linker.
Families **B** (steal a weakly-assigned child) and **C** (joint daughter selection) break that
ceiling but modify existing edges, so they move `adj` and carry the Version-7 edge-FP risk - the
oracle must read their per-specimen `Delta adj`, not only `div_J`.

The machinery is identical: enumerate an atomic edit set per mother, apply to `base_edges`
(remove + add), rescore with `score(...)`, and aggregate exactly as in 0.7/0.8. Fill the two
functions below after reading the family-A ceiling, then reuse Phase 0.8's `agg` / GATE logic.

In [ ]:
# --- Phase 0.9 : families B / C oracle (break the orphan ceiling) ------------
# Family A only recovers divisions whose true D2 the linker left an ORPHAN.
# B/C also reclaim daughters the linker MIS-PARENTED (D2 already has a parent
# P != M). Those edits REMOVE (P,D2) before adding (M,D2), so -- unlike A --
# they can move adj: the oracle must show per-specimen d_adj stays >= 0 (this is
# the Version-7 edge-FP risk, measured directly).
#
#   B (reassign): mother links exactly ONE true daughter; reclaim the OTHER from
#                 whatever parent holds it (orphan => empty removal, so B >= A).
#   C (full):     force BOTH true daughters onto M for EVERY detectable division,
#                 regardless of current linkage (covers M linking to zero / wrong
#                 daughters). This is the absolute detectable ceiling (39/44) --
#                 the most score that division recovery can possibly buy.

def apply_and_score(v, adds, removes):
    rem = set(removes)
    new_edges = [e for e in v["base_edges"] if e not in rem] + list(adds)
    return score(v["coords"], new_edges, v["gt"], v["n_total"])

def _in_edges_to(v, node):
    """base_edges (index space) pointing at `node`."""
    return [(i, j) for (i, j) in v["base_edges"] if j == node]

def family_B_edits(v):
    """Mother links exactly one true daughter; reclaim the OTHER daughter from
    whatever parent currently holds it (orphan => no removal == family A)."""
    g2p, adds, removes, recovered, detectable = v["gt2pred"], [], [], 0, 0
    for m in v["gt_divs"]:
        daughters = v["children"][m][:2]
        if m not in g2p or any(d not in g2p for d in daughters):
            continue
        detectable += 1
        Mi = g2p[m]
        Ds = [g2p[d] for d in daughters]
        linked = [d for d in Ds if d in v["outadj"].get(Mi, [])]
        if len(linked) != 1:
            continue                      # B precondition: exactly one child present
        other = [d for d in Ds if d not in linked][0]
        removes += [e for e in _in_edges_to(v, other) if e[0] != Mi]  # steal from wrong parent
        adds.append((Mi, other))
        recovered += 1
    return adds, removes, detectable, recovered

def family_C_edits(v):
    """Full oracle: for EVERY detectable division force M -> both true daughters,
    removing any non-M incoming edge to either daughter. Absolute ceiling."""
    g2p, adds, removes, recovered, detectable = v["gt2pred"], [], [], 0, 0
    for m in v["gt_divs"]:
        daughters = v["children"][m][:2]
        if m not in g2p or any(d not in g2p for d in daughters):
            continue
        detectable += 1
        Mi = g2p[m]
        for d in daughters:
            Di = g2p[d]
            removes += [e for e in _in_edges_to(v, Di) if e[0] != Mi]
            if Di not in v["outadj"].get(Mi, []):
                adds.append((Mi, Di))
        recovered += 1
    return list(set(adds)), list(set(removes)), detectable, recovered

def run_family(fn):
    rows = []
    for name, v in videos.items():
        adds, removes, detectable, recovered = fn(v)
        orc = apply_and_score(v, adds, removes) if (adds or removes) else v["base"]
        b = v["base"]
        rows.append(dict(
            name=name, specimen=specimen_of(name),
            gt_divs=len(v["gt_divs"]), detectable=detectable, A_edits=recovered,
            d_adj=orc["adj"] - b["adj"], d_divj=orc["div_j"] - b["div_j"],
            d_score=orc["score"] - b["score"],
            base_adj=b["adj"], orc_adj=orc["adj"],
            base_divj=b["div_j"], orc_divj=orc["div_j"],
        ))
    return pl.DataFrame(rows)

famB = run_family(family_B_edits)
famC = run_family(family_C_edits)

for label, df in [("A (orphan)", famA), ("B (reassign)", famB), ("C (full ceiling)", famC)]:
    s = agg(df)
    print(f"\n=== family {label} ===")
    print(s)
    rec, det = df["A_edits"].sum(), df["detectable"].sum()
    d44 = s.filter(pl.col("specimen") == "44b6")["mean_d_score"][0]
    d6b = s.filter(pl.col("specimen") == "6bba")["mean_d_score"][0]
    min_adj = s["mean_d_adj"].min()
    print(f"recovered {rec}/{det} detectable | mean d_score 44b6/6bba = "
          f"{d44:+.4f}/{d6b:+.4f} | min mean_d_adj = {min_adj:+.5f} "
          f"({'adj-safe' if min_adj >= 0 else 'ADJ-NEGATIVE (Version-7 risk)'})")

# GATE 0.9: does B/C buy meaningfully more than A, and stay adj-safe on BOTH?
sA, sC = agg(famA), agg(famC)
inc = (sC.sort('specimen')['mean_d_score'] - sA.sort('specimen')['mean_d_score'])
print("\n=== families B/C read ===")
print(f"full-ceiling gain over A (per specimen, sorted 44b6/6bba): "
      f"{inc.to_list()}")
print("If C's extra gain over A is small -> Phase 1 = orphan-only recovery (simple).")
print("If large & adj-safe -> Phase 1 must reclaim mis-parented daughters (conflict-aware).")


## Optional (Phase-1 prep) - pre-ILP candidate-edge export

Not needed for the Phase-0 gate. When building Phase 1 features we need the transformer scores for
`M -> D2` even though the ILP did NOT select that edge (so it is absent from the saved graph).
Capture them by patching `predict_unet_transformer.predict_video` to persist, per consecutive frame
pair, the full `edge_logits_pair` matrix (raw logit + softmax) BEFORE the candidate/cap filter
(around lines 447-488), keyed by `(gi, gj)`. Save alongside `coords` so Phase 1 can attach
logit / rank / top-1-2 margin features to every candidate triplet.